In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (via IBM Quantum Platform API) Plan. Fitur ini masih dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.

## Gambaran Umum
Dalam kimia kuantum, masalah struktur elektronik berfokus pada pencarian solusi persamaan Schrödinger elektronik — fungsi gelombang kuantum yang menggambarkan perilaku elektron dalam suatu sistem. Fungsi-fungsi gelombang ini adalah vektor amplitudo kompleks, di mana setiap amplitudo merepresentasikan kontribusi dari kemungkinan konfigurasi elektron.

Ground state adalah fungsi gelombang dengan energi terendah dari suatu sistem dan memiliki peran yang sangat penting dalam studi sistem molekuler. Pendekatan paling akurat untuk menghitung ground state mempertimbangkan semua kemungkinan konfigurasi elektron, tetapi ini menjadi tidak praktis untuk sistem yang lebih besar karena jumlah konfigurasi bertumbuh secara eksponensial seiring ukuran sistem.

Handover Iterative Variational Quantum Eigensolver (HI-VQE) adalah metode hibrida kuantum-klasik yang inovatif untuk memperkirakan ground state sistem molekuler secara akurat. Metode ini mengintegrasikan perangkat keras kuantum dengan komputasi klasik, menggunakan prosesor kuantum untuk menjelajahi kandidat konfigurasi elektron secara efisien dan menghitung fungsi gelombang yang dihasilkan pada komputer klasik. Dengan menghasilkan fungsi gelombang yang kompak namun akurat secara kimia, HI-VQE meningkatkan penelitian dan penemuan dalam kimia kuantum dan ilmu material.

![Image showing an overview of Qunova's HI-VQE algorithm](../docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE mengurangi kompleksitas komputasi masalah struktur elektronik dengan memperkirakan ground state secara efisien dengan akurasi tinggi. Metode ini berfokus pada subset konfigurasi elektron yang paling relevan yang dipilih dengan cermat, mengoptimalkan akurasi dan efisiensi sekaligus.

Dengan menggabungkan kekuatan komputer klasik dan kuantum, HI-VQE secara iteratif memperhalus dan meningkatkan estimasi fungsi gelombang saat ini. Teknik konstruksi subspace uniknya membantu membuat pemilihan konfigurasi lebih efisien, sehingga pengguna memiliki kontrol komputasi yang lebih besar dan akurasi yang lebih baik dalam simulasi kimia kuantum.

Kalau kamu ingin mempelajari algoritma ini lebih dalam, kamu bisa [membaca makalah penelitian terkait](https://arxiv.org/abs/2503.06292).
## Deskripsi
Jumlah konfigurasi elektron untuk sistem molekuler bertumbuh secara eksponensial seiring ukuran sistem. Namun, untuk keadaan elektronik tertentu, seperti ground state, umum terjadi bahwa hanya sebagian kecil konfigurasi yang berkontribusi signifikan terhadap energi keadaan tersebut. Metode selected configuration interaction (SCI) memanfaatkan kelangkaan ini untuk mengurangi biaya komputasi dengan mengidentifikasi dan berfokus pada konfigurasi yang paling relevan. Subset konfigurasi ini disebut sebagai subspace.

HI-VQE memanfaatkan efisiensi inheren komputer kuantum dalam merepresentasikan sistem molekuler untuk membantu pencarian subspace. Metode ini mengintegrasikan subrutin klasik dan kuantum untuk memecahkan masalah struktur elektronik dengan akurasi tinggi. Berbeda dengan metode SCI kuantum yang sudah ada, HI-VQE menggabungkan pelatihan variasional, konstruksi subspace iteratif, dan penyaringan konfigurasi pre-diagonalization untuk meningkatkan efisiensi dengan mengurangi pengukuran kuantum, iterasi, dan biaya diagonalisasi klasik. HI-VQE oleh karena itu dapat diterapkan pada sistem molekuler yang lebih besar yang membutuhkan lebih banyak qubit, dan mengurangi biaya untuk memecahkan masalah dengan ukuran tertentu hingga tingkat akurasi yang sama.

![Image showing a detailed description of each step in Qunova's HI-VQE algorithm.](../docs/images/guides/qunova-chemistry/description.avif)

Untuk menghitung ground state suatu sistem, HI-VQE pertama-tama menggunakan paket kimia klasik PySCF untuk menghasilkan representasi molekuler dari input yang diberikan pengguna, seperti geometri molekuler dan informasi molekuler lainnya. Kemudian masuk ke dalam loop optimasi hibrida kuantum-klasik, secara iteratif memperhalus subspace untuk merepresentasikan ground state secara optimal sambil meminimalkan jumlah konfigurasi yang disertakan. Loop berlanjut sampai kriteria konvergensi, seperti ukuran subspace atau stabilitas energi, terpenuhi, setelah itu fungsi gelombang dan energi ground state yang telah dihitung dikeluarkan. Hasil ini dapat digunakan untuk membangun permukaan energi potensial yang akurat dan melakukan analisis lebih lanjut dari sistem tersebut.

Loop optimasi berfokus pada penyesuaian parameter Circuit kuantum untuk menghasilkan subspace berkualitas tinggi. HI-VQE menawarkan tiga opsi Circuit kuantum: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), dan [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Optimasi diinisialisasi dekat dengan keadaan referensi Hartree-Fock karena kesesuaiannya yang umum. Circuit kemudian dieksekusi pada perangkat kuantum dan konfigurasi disampel dari keadaan kuantum yang dihasilkan sebelum dikembalikan sebagai string biner. Karena noise perangkat kuantum, beberapa konfigurasi yang disampel mungkin tidak valid secara fisik, gagal menjaga jumlah elektron atau spin. HI-VQE mengatasi ini menggunakan proses pemulihan konfigurasi dari paket [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), sehingga pengguna dapat mengoreksi konfigurasi yang tidak valid atau membuangnya.

Konfigurasi yang valid kemudian menjalani langkah penyaringan opsional untuk menghapus konfigurasi yang diprediksi berkontribusi minimal. Ini mengurangi dimensi subspace, sehingga menurunkan biaya langkah diagonalisasi. Jika penyaringan diaktifkan, maka Hamiltonian subspace awal dikonstruksi dari konfigurasi yang valid dan diagonalisasi dilakukan dengan kriteria penghentian yang sangat longgar. Meskipun akurasi amplitudo yang dihasilkan untuk setiap konfigurasi rendah, cara ini efektif untuk memprediksi konfigurasi mana yang harus dikeluarkan dari subspace pada iterasi ini, dan cepat untuk dihitung.

Konfigurasi yang dipilih ditambahkan ke subspace, dan Hamiltonian sistem diproyeksikan ke dalam subspace ini. Subspace diperbarui secara iteratif, mempertahankan konfigurasi yang paling relevan di seluruh iterasi. Pendekatan ini kontras dengan metode alternatif karena Circuit kuantum tidak perlu memperkirakan keseluruhan ground state di setiap langkah.

Selanjutnya, Hamiltonian subspace didiagonalisasi secara klasik untuk mendapatkan eigenvalue terendah dan eigenvector yang sesuai, merepresentasikan perkiraan ground state dan energinya. Saat kualitas subspace meningkat melalui iterasi, ground state yang dihitung semakin mendekati ground state sebenarnya. Langkah penyaringan tambahan dapat dilakukan pada titik ini untuk menghapus konfigurasi dari subspace yang tidak memiliki kontribusi substansial terhadap ground state yang dihitung. Langkah ini memastikan bahwa subspace yang dibawa ke iterasi berikutnya sekompak mungkin. Ini dievaluasi berdasarkan amplitudo yang dikembalikan oleh diagonalisasi, karena ini merepresentasikan kontribusi kepentingan setiap konfigurasi terhadap ground state yang dihitung.

Pemeriksaan konvergensi kemudian menentukan apakah pelatihan lebih lanjut akan meningkatkan hasil. Jika ya, maka langkah ekspansi klasik opsional dilakukan, parameter Circuit kuantum diperbarui untuk lebih meminimalkan energi yang dihitung, dan proses berulang. Langkah ekspansi klasik menghasilkan konfigurasi tambahan untuk subspace, melengkapi konfigurasi yang disampel dari perangkat kuantum. Pertama-tama mengidentifikasi konfigurasi dengan amplitudo terbesar dalam hasil diagonalisasi, sebelum menghasilkan konfigurasi baru dengan eksitasi tunggal dan ganda dari konfigurasi yang diidentifikasi. Jumlah konfigurasi yang diinginkan kemudian ditambahkan ke subspace.

Setelah ditentukan bahwa iterasi telah konvergen, HI-VQE mengembalikan ground state yang dihitung (dalam bentuk keadaan dalam subspace dan amplitudunya dalam fungsi gelombang ground state), energinya, dan ukuran variansi energi yang memberikan indikasi apakah keadaan yang dihitung membentuk eigenstate dari Hamiltonian sistem.

Pengguna dapat memutuskan Circuit kuantum yang digunakan dan jumlah shot yang diambil untuk setiap Circuit kuantum, serta mengontrol ukuran subspace atau mengaktifkan generasi klasik konfigurasi tambahan untuk membantu konfigurasi yang dihasilkan kuantum. Dengan cara ini pengguna dapat menyesuaikan perilaku HI-VQE sesuai dengan aplikasi yang mereka inginkan.
## Memulai
Pertama, [minta akses ke fungsi ini](https://forms.office.com/r/zN3hvMTqJ1).
Kemudian, autentikasi menggunakan [API key IBM Quantum&reg;](http://quantum.cloud.ibm.com/) kamu dan, dengan asumsi kamu sudah [menyimpan akun](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokalmu, pilih Qiskit Function sebagai berikut:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Input
Lihat tabel berikut untuk semua parameter input yang diterima fungsi. Bagian [Opsi Molekul](#molecule-options) dan [Opsi HI-VQE](#hi-vqe-options) berikutnya memberikan detail lebih lanjut tentang argumen-argumen tersebut.
| Nama                   | Tipe                                                           | Deskripsi                                                                                                                                                                                                                                                                                                 | Wajib | Default                                                                  | Contoh                                                                                   |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Ini bisa berupa string atau daftar terstruktur yang berisi pasangan atom dan koordinat. Jika diberikan sebagai string, harus berupa geometri molekuler dalam Format Koordinat Kartesian. Jika diberikan sebagai daftar, harus berupa daftar dari daftar yang masing-masing berisi string atom dan tuple koordinat. | Ya      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` atau `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Nama Backend untuk membuat kueri.                                                                                                                                                                                                                                                                      | Ya      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Dimensi subspace maksimum untuk diagonalisasi. Lebih sedikit keadaan akan digunakan jika jumlahnya bukan bilangan kuadrat sempurna.                                                                                                                                                                                                                                                    | Ya      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Jumlah maksimum keadaan CI yang dihasilkan secara klasik untuk disertakan dalam setiap iterasi.                                                                                                                                                                                                                     | Ya      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Opsi terkait molekul yang digunakan sebagai input ke HI-VQE. Lihat bagian [Opsi Molekul](#molecule-options) untuk detail lebih lanjut.                                                                                                                                                                                                                                                | Tidak       | Lihat bagian [Opsi Molekul](#molecule-options) untuk detail.                                 | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Opsi yang mengontrol perilaku algoritma HI-VQE. Lihat bagian [Opsi HI-VQE](#hi-vqe-options) untuk detail lebih lanjut.                                                                                                                                                                                                                                                | Tidak       | Lihat bagian [Opsi HI-VQE](#hi-vqe-options) untuk detail.                                 | `{"shots": 10_000, "max_iter": 10 }`                               |
### Opsi Molekul
Tabel berikut merinci semua kunci dan nilai yang dapat diatur dalam kamus `molecule_options`, beserta tipe data dan nilai defaultnya. Semua kunci bersifat opsional.

| Kunci                               | Tipe nilai                          | Nilai Default                    | Rentang valid                                                                                                                                                    | Penjelasan                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Berbagai                                                                                                                                                        | Bilangan bulat yang menentukan total muatan bersih sistem molekuler. Nilai default adalah 0; namun, bisa berupa bilangan bulat apa pun.                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Berbagai                                                                                                                                                        | String yang menentukan tipe basis; ini diteruskan ke pyscf. (mis: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"active_orbitals"`               | `List[int]`                         | Setiap indeks orbital.             | Indeks orbital spasial yang valid untuk masalah tersebut.                                                                                                             | Daftar indeks orbital aktif dalam interval [0, n) di mana n adalah jumlah qubit yang digunakan dalam masalah. Jika ini ditentukan, maka argumen frozen_orbitals juga harus ditentukan.                                                                                                                                                                                                                                                                                                                                            |
| `"frozen_orbitals"`               | `List[int]`                         | Tidak ada indeks.                      | Indeks orbital spasial yang valid untuk masalah tersebut, tidak termasuk orbital aktif.                                                                                  | Daftar indeks orbital beku dalam rentang yang sama dengan active_orbitals. Jika ditentukan, maka active_orbitals juga harus ditentukan. Perhatikan bahwa hanya orbital yang terisi yang harus dibekukan karena jumlah elektron aktif berkurang 2 untuk setiap orbital yang terisi yang dibekukan.                                                                                                                                                                                                                                        |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Orbital molekuler Hartree-Fock. | Berbagai.                                                                                                                                                       | Koefisien untuk orbital spasial yang digunakan dalam penghitungan integral tolakan elektronik untuk sistem. Beberapa contoh yang valid adalah orbital molekuler Hartree-Fock, orbital alami, dan orbital AVAS.                                                                                                                                                                                                                                                                                                                   |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` atau `False`                                                                                                                                              | Digunakan untuk memanggil simetri kelompok titik untuk perhitungan molekuler awal guna membangun basis orbital yang disesuaikan dengan simetri. Orbital yang disesuaikan dengan simetri ini digunakan sebagai fungsi basis untuk perhitungan SCF berikutnya. Nilai default adalah False; jika diatur ke True, maka akan dipanggil dan kelompok titik arbitrer akan secara otomatis terdeteksi dan digunakan. Jika simetri tertentu ditetapkan, misalnya, symmetry = "Dooh", maka error akan dimunculkan jika geometri molekuler tidak tunduk pada simetri yang diperlukan ini. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Dapat digunakan untuk menghasilkan subgroup dari simetri yang terdeteksi. Ini tidak berpengaruh ketika simetri ditentukan menggunakan argumen kata kunci symmetry.                                                                                                                                                                                                                                                                                                                                                         |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan satuan pengukuran yang digunakan untuk koordinat dan jarak atom. Default-nya adalah menggunakan satuan angstrom.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan model nuklir yang akan digunakan. Secara default menggunakan model nuklir titik, dan nilai lainnya mengaktifkan model nuklir Gaussian. Jika fungsi diberikan, fungsi tersebut akan digunakan dengan model nuklir Gaussian untuk menghasilkan nilai distribusi muatan nuklir 'zeta'.                                                                                                                                                                                                                                                   |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan pseudopotensial untuk atom-atom dalam molekul. Ini adalah None secara default, menunjukkan bahwa tidak ada pseudopotensial yang diterapkan dan semua elektron secara eksplisit disertakan dalam perhitungan.                                                                                                                                                                                                                                                                                                                                                |
| `"cart"`                          | `bool`                              | `False`                          | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Menentukan apakah akan menggunakan GTO kartesian sebagai fungsi basis momentum sudut dalam perhitungan. Nilai default False akan menggunakan GTO sferis.                                                                                                                                                                                                                                                                                                                                               |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Lihat [dokumentasi pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                                      | Mengatur momen magnetik spin kolinear dari setiap atom. Secara default, ini adalah None dan setiap atom diinisialisasi dengan spin nol.                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | mis. ["H 1s", "O 2p"] untuk H2O                                                                                                                                                             | Ini mendefinisikan Orbital Atom yang akan disertakan dalam skema AVAS. Lihat [dokumentasi AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Antara 0.0 dan 2.0                                                                                                                                                      | Ini menentukan nilai cutoff yang digunakan untuk menentukan Orbital Atom (AO) mana yang dipertahankan dalam ruang aktif.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` atau `"ccsd"`                                                                                                                                            | Ini mendefinisikan pendekatan teoritis untuk menyiapkan orbital alami dan memilih orbital aktif berdasarkan skema Natural Orbital Occupation Numbers (NOONs). Lihat [dokumentasi NOONs](https://doi.org/10.1063/5.0042147). Baik indeks orbital aktif maupun yang dibekukan harus disediakan untuk mengontrol jumlah orbital (dan jumlah qubit).                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
### Opsi HI-VQE
Tabel berikut merinci semua kunci dan nilai yang dapat diatur dalam kamus `hivqe_options`, beserta tipe data dan nilai defaultnya. Semua kunci bersifat opsional.

| Kunci                               | Tipe nilai                          | Nilai Default                    | Rentang valid                                                                                                                                                    | Penjelasan                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Antara 1 dan 10.000.                                                                                                                                          | Jumlah shot yang digunakan pada perangkat kuantum per iterasi.                                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| `"max_iter"`                      | `int`                               | `25`                             | Antara 1 dan 50.                                                                                                                                              | Jumlah maksimum iterasi yang dijalankan untuk mengoptimalkan ansatz. Algoritma mungkin menggunakan lebih sedikit iterasi jika konvergensi tercapai lebih awal.                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| `"initial_basis_states"`          | `List[str]`                         | Keadaan Hartree-Fock.          | String biner dengan jumlah bit yang sesuai dengan jumlah qubit yang diperlukan untuk masalah tersebut.                                                                 | Dapat digunakan untuk memulai ulang algoritma dengan keadaan klasik dari hasil sebelumnya.                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, atau `"lucj"`                                                                                                                                  | Ini menentukan ansatz kuantum yang akan dioptimalkan untuk menghasilkan keadaan baru. `"epa"` memilih excitation-preserving ansatz. `"hea"` memilih hardware-efficient ansatz. `"lucj"` memilih local unitary cluster Jastrow ansatz.                                                                                                                                                                                                                                                                                                       |
| `"convergence_count"`             | `int`                               | `3`                              | Minimal 2.                                                                                                                                                    | Jumlah iterasi tanpa perubahan signifikan pada energi yang dihitung yang harus berlalu sebelum algoritma dianggap telah konvergen.                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Lebih dari 0 dan paling besar 1.                                                                                                                                     | Besarnya perubahan energi yang dihitung yang dianggap signifikan untuk tujuan pemeriksaan konvergensi.                                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` atau `False`                                                                                                                                              | Jika `True`, iterasi `convergence_count` harus terjadi tanpa perubahan signifikan yang menginterupsi untuk memenuhi syarat sebagai konvergen. Jika `False`, maka algoritma akan berhenti setelah `convergence_count` jika perubahan tidak signifikan telah terjadi pada iterasi mana pun selama proses optimasi.                                                                                                                                                                                                                                                 |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` atau `False`.                                                                                                                                             | Apakah akan menggunakan pemulihan konfigurasi dari paket `qiskit-addon-sqd` atau tidak. Jika True, keadaan tidak valid yang disampel dari perangkat kuantum dikoreksi secara klasik. Jika False, keadaan tersebut dibuang.                                                                                                                                                                                                                                                                                                                                      |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Salah satu dari `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, atau `"sca"`. Jika menggunakan ansatz `"lucj"`, `"lucj_default"` juga merupakan opsi. | Ini menentukan skema entanglement yang harus digunakan dalam Circuit kuantum, mengikuti konvensi umum Qiskit dan [ffsim untuk ansatz LUCJ](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                       |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Lebih dari 0.                                                                                                                                                | Jumlah pengulangan setiap lapisan dalam Circuit kuantum.                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Minimal 0, dan kurang dari 1.                                                                                                                                   | Toleransi untuk memutuskan keadaan mana yang harus disaring dari subspace setelah diagonalisasi. Ini menentukan ambang penyertaan untuk keadaan subspace berdasarkan amplitudo yang dihitung.                                                                                                                                                                                                                                                                                                                                  |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Antara `1e-4` dan `1e-1`, inklusif.                                                                                                                          | Toleransi untuk memprediksi keadaan mana yang harus disaring dari subspace sebelum diagonalisasi. Ini mengontrol akurasi amplitudo yang diprediksi untuk setiap keadaan, dengan nilai yang lebih kecil menghasilkan prediksi yang lebih akurat.                                                                                                                                                                                                                                                                            |
## Output
Fungsi ini mengembalikan kamus dengan empat kunci dan nilai. Kunci dan nilai didokumentasikan dalam tabel berikut:

| Kunci             | Tipe Nilai    | Penjelasan                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Perkiraan energi ground state molekul.                                                                      |
| `"states"`      | `List[str]`   | Determinan yang dipilih yang membentuk subspace yang digunakan untuk menyelesaikan energi. Dalam format alpha-beta bergantian. |
| `"eigenvector"` | `List[float]` | Eigenvector yang bersesuaian dengan ground state dari subspace yang terdiri dari `"states"`.                                 |
| `"energy_variance"` | `float` | Variansi energi ground state dari subspace yang terdiri dari `"states"`, memberikan indikasi kualitas solusi. Nilai ini non-negatif dan nilai yang lebih rendah berarti ground state dari subspace lebih mendekati eigenstate dari Hamiltonian sistem. |
| `"energy_history"` | `List[float]` | Energi yang dihitung setiap iterasi selama proses optimasi hibrida, dalam urutan yang sama saat dihitung. Dua energi dihitung per iterasi sebagai bagian dari proses optimasi SPSA. |
## Contoh
Contoh pertama menunjukkan cara menghitung energi ground state molekul NH3 menggunakan algoritma HI-VQE.
#### Mendefinisikan geometri molekuler dan opsi
Geometri molekuler NH3 diberikan dengan koordinat Kartesian yang dipisahkan dengan ";" untuk setiap atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Opsi tambahan dapat didefinisikan dan disediakan untuk sistem molekuler dalam format kamus berikut.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Eksekusi fungsi dengan input geometri dan opsi.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Sebaiknya cetak ID job Fungsi agar dapat diberikan dalam permintaan dukungan jika ada yang tidak beres.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Contoh ini kemudian menggunakan 16 qubit dengan 8 orbital basis sto3g untuk molekul NH3.
Periksa [status](/guides/functions#check-job-status) beban kerja Qiskit Function kamu atau ambil [hasil](/guides/functions#retrieve-results) sebagai berikut:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Setelah job selesai, hasilnya dapat diperoleh dengan instance `result()`.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Untuk mengakses energi ground state, gunakan kunci "energy". Kunci "eigenvector" memberikan koefisien CI dengan notasi bitstring yang sesuai dari konfigurasi elektron yang tersimpan dengan "states" dari hasilnya.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Performa
Bagian ini menunjukkan perhitungan benchmark yang telah didemonstrasikan dari HI-VQE dengan kasus 24-qubit untuk Li2S, kasus 40-qubit untuk molekul N2, dan kasus 44-qubit untuk sistem FeP-NO.
#### Kurva permukaan energi potensial disosiasi untuk molekul Li2S dengan 24 qubit
Kurva PES ditampilkan dengan referensi FCI dan tebakan awal dari RHF, bersama dengan kesalahan energi dari referensi FCI.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Perhitungan telah dilakukan dengan geometri dan opsi berikut.